In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

CLEAN_ROOT = Path("/home/aniket/Programming/renters/data/cmhc-rental-clean")
pct_dir = CLEAN_ROOT / "pct-change-rent"
vac_dir = CLEAN_ROOT / "vacancy-rate"

# Build long-format panel: city × year with rent_pct and vacancy
records = []
for pct_path in sorted(pct_dir.glob("*.csv")):
    city = pct_path.stem
    vac_path = vac_dir / pct_path.name
    if not vac_path.exists():
        continue
    pct_df = pd.read_csv(pct_path)[["Year", "Total"]].rename(columns={"Total": "rent_pct"})
    vac_df  = pd.read_csv(vac_path)[["Year", "Total"]].rename(columns={"Total": "vacancy"})
    merged = pct_df.merge(vac_df, on="Year", how="inner")
    merged["city"] = city
    records.append(merged)

panel = pd.concat(records, ignore_index=True)
panel = panel.dropna(subset=["rent_pct", "vacancy"])

# Label periods: pre-rent-control-weakening (2011–2017) vs post (2019–2025)
# 2018 is excluded as it is the transition year
panel["period"] = np.where(
    panel["Year"].between(2011, 2017), "pre_2018",
    np.where(panel["Year"].between(2019, 2025), "post_2018", "other")
)

print(f"Panel: {len(panel)} city-year obs | {panel['city'].nunique()} cities | {panel['Year'].min()}–{panel['Year'].max()}")
panel.head(8)

Panel: 900 city-year obs | 30 cities | 1991–2025


,Year,rent_pct,vacancy,city,period
1,1991,4.8,1.8,ajax,other
2,1992,4.1,1.7,ajax,other
3,1993,3.3,1.7,ajax,other
4,1994,0.0,0.4,ajax,other
5,1995,0.9,0.4,ajax,other
6,1996,3.5,1.1,ajax,other
7,1997,0.0,0.9,ajax,other
8,1998,8.4,0.0,ajax,other


In [9]:

# ── Cell 2: Pearson correlations — with confidence intervals + formal significance ──
def pearson_ci(x, y, alpha=0.05):
    """Pearson r with Fisher-z 95% CI."""
    r, p = stats.pearsonr(x, y)
    n = len(x)
    se_z   = 1 / np.sqrt(n - 3)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    ci_lo  = np.tanh(np.arctanh(r) - z_crit * se_z)
    ci_hi  = np.tanh(np.arctanh(r) + z_crit * se_z)
    return r, p, ci_lo, ci_hi, n

pre  = panel[panel["period"] == "pre_2018"]
post = panel[panel["period"] == "post_2018"]

r_all,  p_all,  lo_all,  hi_all,  n_all  = pearson_ci(panel["vacancy"], panel["rent_pct"])
r_pre,  p_pre,  lo_pre,  hi_pre,  n_pre  = pearson_ci(pre["vacancy"],   pre["rent_pct"])
r_post, p_post, lo_post, hi_post, n_post = pearson_ci(post["vacancy"],  post["rent_pct"])

print("Vacancy → Rent Growth  |  Pearson r with 95% CI (Fisher's z-transformation)")
print("─" * 78)
for label, r, p, lo, hi, n in [
    ("All years",  r_all,  p_all,  lo_all,  hi_all,  n_all),
    ("Pre-2018",   r_pre,  p_pre,  lo_pre,  hi_pre,  n_pre),
    ("Post-2018",  r_post, p_post, lo_post, hi_post, n_post),
]:
    sig = "✓ sig" if p < 0.05 else "✗ n.s."
    print(f"  {label:10s}  r={r:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
          f"R²={r**2:.3f} ({r**2*100:.1f}%)  p={p:.4f}  {sig}  n={n}")

# Fisher's z-test: formally test whether the pre and post correlations differ from each other
z_diff = (np.arctanh(r_post) - np.arctanh(r_pre)) / np.sqrt(1/(n_pre - 3) + 1/(n_post - 3))
p_diff = 2 * (1 - stats.norm.cdf(abs(z_diff)))
print(f"\nFisher's z-test (pre r vs post r):  z={z_diff:.3f},  p={p_diff:.4f}"
      f"  → {'significantly different' if p_diff < 0.05 else 'NOT significantly different'} at α=0.05")

print()
print("▶  Key finding:")
print(f"   • Pre-2018:  vacancy explains only {r_pre**2*100:.1f}% of rent growth variance (p={p_pre:.3f}, n.s.)")
print(f"   • Post-2018: vacancy explains only {r_post**2*100:.1f}% of rent growth variance (p={p_post:.3f})")
print(f"   • The pre/post correlations are statistically indistinguishable (p={p_diff:.3f}).")
print(f"   • The overall r={r_all:.3f} is a cross-era artefact: over 30 years, vacancy has")
print(f"     broadly fallen while rents have risen, producing a spurious long-run correlation")
print(f"     that does not reflect a meaningful within-period causal mechanism.")


Vacancy → Rent Growth  |  Pearson r with 95% CI (Fisher's z-transformation)
──────────────────────────────────────────────────────────────────────────────
  All years   r=-0.337  95% CI [-0.394, -0.278]  R²=0.114 (11.4%)  p=0.0000  ✓ sig  n=900
  Pre-2018    r=-0.125  95% CI [-0.258, +0.013]  R²=0.016 (1.6%)  p=0.0753  ✗ n.s.  n=203
  Post-2018   r=-0.173  95% CI [-0.313, -0.025]  R²=0.030 (3.0%)  p=0.0224  ✓ sig  n=175

Fisher's z-test (pre r vs post r):  z=-0.467,  p=0.6404  → NOT significantly different at α=0.05

▶  Key finding:
   • Pre-2018:  vacancy explains only 1.6% of rent growth variance (p=0.075, n.s.)
   • Post-2018: vacancy explains only 3.0% of rent growth variance (p=0.022)
   • The pre/post correlations are statistically indistinguishable (p=0.640).
   • The overall r=-0.337 is a cross-era artefact: over 30 years, vacancy has
     broadly fallen while rents have risen, producing a spurious long-run correlation
     that does not reflect a meaningful within-period causa

In [10]:

# ── Cell 3: OLS regression vacancy → rent_pct, per period — with slope CIs ────
def ols_summary(df, label):
    slope, intercept, r, p, se_slope = stats.linregress(df["vacancy"], df["rent_pct"])
    n      = len(df)
    t_crit = stats.t.ppf(0.975, df=n - 2)
    lo, hi = slope - t_crit * se_slope, slope + t_crit * se_slope
    print(f"  {label:12s} | slope={slope:+.3f} [{lo:+.3f}, {hi:+.3f}]"
          f"  intercept={intercept:.2f}  R²={r**2:.3f}  p={p:.4f}")
    return dict(period=label, slope=round(slope, 3),
                slope_ci_lo=round(lo, 3), slope_ci_hi=round(hi, 3),
                intercept=round(intercept, 3), r2=round(r**2, 3), p=round(p, 4), n=n)

print("OLS: rent_pct ~ vacancy  (slope 95% CI in brackets)")
print("─" * 75)
rows = []
rows.append(ols_summary(panel, "all_years"))
rows.append(ols_summary(pre,   "pre_2018"))
rows.append(ols_summary(post,  "post_2018"))
reg_overall = pd.DataFrame(rows)

icp_pre, icp_post = rows[1]["intercept"], rows[2]["intercept"]
print()
print("▶  Intercept interpretation (expected rent growth when vacancy = 0, i.e. structural baseline):")
print(f"   Pre-2018:  {icp_pre:.2f}%/yr  — baseline annual rent increase")
print(f"   Post-2018: {icp_post:.2f}%/yr  — baseline annual rent increase  (+{icp_post - icp_pre:.2f} pp structural shift)")
print()
print("   The slope (vacancy's marginal effect on rent) barely changed between periods.")
print("   The intercept nearly doubled — consistent with a structural upward shift in rent")
print("   growth that is independent of vacancy, not an intensified vacancy-driven mechanism.")
reg_overall


OLS: rent_pct ~ vacancy  (slope 95% CI in brackets)
───────────────────────────────────────────────────────────────────────────
  all_years    | slope=-0.397 [-0.469, -0.324]  intercept=4.04  R²=0.114  p=0.0000
  pre_2018     | slope=-0.126 [-0.264, +0.013]  intercept=2.96  R²=0.016  p=0.0753
  post_2018    | slope=-0.441 [-0.818, -0.063]  intercept=6.05  R²=0.030  p=0.0224

▶  Intercept interpretation (expected rent growth when vacancy = 0, i.e. structural baseline):
   Pre-2018:  2.96%/yr  — baseline annual rent increase
   Post-2018: 6.05%/yr  — baseline annual rent increase  (+3.09 pp structural shift)

   The slope (vacancy's marginal effect on rent) barely changed between periods.
   The intercept nearly doubled — consistent with a structural upward shift in rent
   growth that is independent of vacancy, not an intensified vacancy-driven mechanism.


,period,slope,slope_ci_lo,slope_ci_hi,intercept,r2,p,n
0,all_years,-0.397,-0.469,-0.324,4.040,0.114,0.0000,900
1,pre_2018,-0.126,-0.264,0.013,2.958,0.016,0.0753,203
2,post_2018,-0.441,-0.818,-0.063,6.048,0.030,0.0224,175


In [12]:

# ── Cell 4: Per-city regression slope — pre vs post 2018 ──────────────────────
city_reg_rows = []
for city, grp in panel.groupby("city"):
    for period_label, period_mask in [("pre_2018",  grp["period"] == "pre_2018"),
                                       ("post_2018", grp["period"] == "post_2018")]:
        sub = grp[period_mask].dropna(subset=["vacancy", "rent_pct"])
        if len(sub) < 4:
            continue
        slope, intercept, r, p, _ = stats.linregress(sub["vacancy"], sub["rent_pct"])
        city_reg_rows.append(dict(city=city, period=period_label,
                                  slope=round(slope, 3), r2=round(r**2, 3),
                                  p=round(p, 4), n=len(sub)))

city_reg = pd.DataFrame(city_reg_rows)

# Pivot slopes for side-by-side comparison; pull post-2018 p-values for significance flags
slope_piv = city_reg.pivot(index="city", columns="period", values="slope").reset_index()
slope_piv.columns.name = None
slope_piv = slope_piv.rename(columns={"pre_2018": "slope_pre", "post_2018": "slope_post"})

p_post_map = city_reg[city_reg["period"] == "post_2018"].set_index("city")["p"]
pivot_comp = slope_piv.dropna(subset=["slope_pre", "slope_post"]).copy()
pivot_comp["p_post"]        = pivot_comp["city"].map(p_post_map)
pivot_comp["slope_flipped"] = (pivot_comp["slope_pre"] < 0) & (pivot_comp["slope_post"] > 0)
pivot_comp["sig_pos_post"]  = (pivot_comp["slope_post"] > 0) & (pivot_comp["p_post"] < 0.05)

n_total   = len(pivot_comp)
n_neg_pre = int((pivot_comp["slope_pre"]  < 0).sum())
n_neg_post= int((pivot_comp["slope_post"] < 0).sum())
n_flipped = int(pivot_comp["slope_flipped"].sum())
n_sig_pos = int(pivot_comp["sig_pos_post"].sum())

# Wilcoxon signed-rank test on per-city slope changes — did slopes shift systematically?
slope_diffs = pivot_comp["slope_post"].values - pivot_comp["slope_pre"].values
w_stat, w_p = stats.wilcoxon(slope_diffs)

print(f"Cities with data in both periods: {n_total}")
print()
print(f"Cities with negative slope (vacancy ↑ → rent growth ↓, as expected):")
print(f"  Pre-2018:  {n_neg_pre}/{n_total}  ({100*n_neg_pre/n_total:.0f}%)")
print(f"  Post-2018: {n_neg_post}/{n_total}  ({100*n_neg_post/n_total:.0f}%)")
print(f"  Slope flipped negative→positive (link fully reversed): {n_flipped}/{n_total}  ({100*n_flipped/n_total:.0f}%)")
print(f"  Significantly positive slope post-2018: {n_sig_pos}/{n_total}  (stat. confirmed reversals)")
print()
print(f"Wilcoxon signed-rank test on per-city slope changes (H₀: no systematic shift):")
print(f"  W={w_stat:.1f},  p={w_p:.4f}  → {'significant shift' if w_p < 0.05 else 'no systematic shift in slopes'}")
print()
print("▶  The per-city picture matches the aggregate: the vacancy-rent slope did not")
print("   change direction consistently across cities. What changed is the intercept")
print("   (baseline rent pressure), not how individual cities respond to vacancy.")

pivot_comp.drop(columns=["p_post"]).sort_values("slope_post", ascending=False)


Cities with data in both periods: 28

Cities with negative slope (vacancy ↑ → rent growth ↓, as expected):
  Pre-2018:  15/28  (54%)
  Post-2018: 18/28  (64%)
  Slope flipped negative→positive (link fully reversed): 7/28  (25%)
  Significantly positive slope post-2018: 0/28  (stat. confirmed reversals)

Wilcoxon signed-rank test on per-city slope changes (H₀: no systematic shift):
  W=133.0,  p=0.1145  → no systematic shift in slopes

▶  The per-city picture matches the aggregate: the vacancy-rent slope did not
   change direction consistently across cities. What changed is the intercept
   (baseline rent pressure), not how individual cities respond to vacancy.


,city,slope_post,slope_pre,slope_flipped,sig_pos_post
5,chatham-kent,1.348,-0.343,True,False
29,windsor,1.185,-0.328,True,False
7,greater sudbury,1.139,-0.525,True,False
4,burlington,0.648,0.490,False,False
13,london,0.585,-0.504,True,False
22,peterborough,0.373,-0.077,True,False
27,waterloo,0.366,0.773,False,False
19,oakville,0.356,-0.740,True,False
18,niagara falls,0.287,-0.377,True,False
14,markham,0.182,0.733,False,False


In [13]:

# ── Cell 5: "Rents rose faster despite stable vacancy" test ───────────────────
# Stable vacancy = |mean post − mean pre| ≤ 0.5pp;  "faster" = higher mean rent growth post-2018

period_means = (panel[panel["period"].isin(["pre_2018", "post_2018"])]
                .groupby(["city", "period"])[["rent_pct", "vacancy"]].mean().reset_index())
pm_pivot = period_means.pivot(index="city", columns="period", values=["rent_pct", "vacancy"])
pm_pivot.columns = ["_".join(c) for c in pm_pivot.columns]
pm_pivot = pm_pivot.reset_index().dropna()

pm_pivot["vac_change"]      = pm_pivot["vacancy_post_2018"]  - pm_pivot["vacancy_pre_2018"]
pm_pivot["rent_pct_change"] = pm_pivot["rent_pct_post_2018"] - pm_pivot["rent_pct_pre_2018"]
pm_pivot["vac_stable"]      = pm_pivot["vac_change"].abs() <= 0.5
pm_pivot["rent_faster"]     = pm_pivot["rent_pct_change"] > 0

n_eligible      = int(pm_pivot["vac_stable"].sum())
n_faster_stable = int((pm_pivot["vac_stable"] & pm_pivot["rent_faster"]).sum())
n_faster_all    = int(pm_pivot["rent_faster"].sum())
n_cities        = len(pm_pivot)

# Binomial tests — H₀: without a structural change, 50% of cities would show faster growth
binom_stable = stats.binomtest(n_faster_stable, n_eligible, 0.5, alternative="greater")
binom_all    = stats.binomtest(n_faster_all,    n_cities,   0.5, alternative="greater")

print(f"Cities with stable vacancy (|Δ| ≤ 0.5pp):  {n_eligible}/{n_cities}")
print(f"  Of those, faster rent growth post-2018:   {n_faster_stable}/{n_eligible}"
      f"  ({100*n_faster_stable/n_eligible:.0f}%)")
print(f"  Binomial test (H₀: 50%): p = {binom_stable.pvalue:.4f}")
print()
print(f"Among ALL cities (regardless of how vacancy moved):")
print(f"  Faster rent growth post-2018: {n_faster_all}/{n_cities}  ({100*n_faster_all/n_cities:.0f}%)")
print(f"  Binomial test (H₀: 50%): p = {binom_all.pvalue:.6f}")
print()
print("▶  Interpretation: Rents accelerated almost universally regardless of what vacancy did.")
print("   Both binomial tests confirm this cannot plausibly be attributed to chance.")

pm_pivot[["city", "vacancy_pre_2018", "vacancy_post_2018", "vac_change",
          "rent_pct_pre_2018", "rent_pct_post_2018", "rent_pct_change",
          "vac_stable", "rent_faster"]].round(2)


Cities with stable vacancy (|Δ| ≤ 0.5pp):  14/30
  Of those, faster rent growth post-2018:   13/14  (93%)
  Binomial test (H₀: 50%): p = 0.0009

Among ALL cities (regardless of how vacancy moved):
  Faster rent growth post-2018: 28/30  (93%)
  Binomial test (H₀: 50%): p = 0.000000

▶  Interpretation: Rents accelerated almost universally regardless of what vacancy did.
   Both binomial tests confirm this cannot plausibly be attributed to chance.


,city,vacancy_pre_2018,vacancy_post_2018,vac_change,rent_pct_pre_2018,rent_pct_post_2018,rent_pct_change,vac_stable,rent_faster
0,ajax,2.53,1.50,-1.03,2.15,3.55,1.40,False,True
1,barrie,2.31,2.84,0.53,2.74,4.80,2.06,False,True
2,brampton,1.67,2.44,0.77,2.37,4.47,2.10,False,True
3,brantford,2.26,1.93,-0.33,2.98,6.65,3.67,True,True
4,burlington,1.34,1.64,0.30,3.76,4.77,1.01,True,True
5,chatham-kent,4.37,2.62,-1.75,2.07,6.28,4.20,False,True
6,clarington,0.87,0.43,-0.44,2.20,4.87,2.67,True,True
7,greater sudbury,3.82,1.46,-2.36,2.52,8.74,6.22,False,True
8,guelph,1.24,1.90,0.66,2.93,5.30,2.37,False,True
9,hamilton,3.63,3.14,-0.49,3.39,5.14,1.76,True,True


In [14]:

# ── Cell 6: Robustness checks — effect size, non-parametric test, thresholds ──

# Threshold sensitivity: does the ">0.5pp stable vacancy" finding hold at tighter/looser cutoffs?
print("Threshold sensitivity (stable vacancy definition):")
print(f"{'Threshold':>10} | {'Cities stable':>14} | {'Rents faster':>13} | {'%':>4} | {'Binom. p':>9}")
print("─" * 62)
threshold_rows = []
for thresh in [0.3, 0.5, 1.0]:
    stable = pm_pivot[pm_pivot["vac_change"].abs() <= thresh]
    faster = stable[stable["rent_faster"]]
    pct    = 100 * len(faster) / len(stable) if len(stable) > 0 else float("nan")
    bp     = stats.binomtest(len(faster), len(stable), 0.5, alternative="greater").pvalue
    print(f"  ≤ {thresh:.1f}pp    |     {len(stable):3d}         |     {len(faster):3d}       "
          f"| {pct:.0f}%  | {bp:.4f}")
    threshold_rows.append(dict(threshold=thresh, n_stable=len(stable),
                               n_faster=len(faster), pct_faster=round(pct, 1),
                               binom_p=round(bp, 4)))
thresh_df = pd.DataFrame(threshold_rows)

# Mean values
mean_pre  = pm_pivot["rent_pct_pre_2018"].mean()
mean_post = pm_pivot["rent_pct_post_2018"].mean()
diffs     = pm_pivot["rent_pct_post_2018"] - pm_pivot["rent_pct_pre_2018"]

# Paired t-test (parametric)
t_stat, t_p = stats.ttest_rel(pm_pivot["rent_pct_post_2018"], pm_pivot["rent_pct_pre_2018"])
# Wilcoxon signed-rank test (non-parametric — no normality assumption required)
w_stat, w_p = stats.wilcoxon(pm_pivot["rent_pct_post_2018"], pm_pivot["rent_pct_pre_2018"])
# Cohen's d (standardised mean difference — practical effect size)
cohens_d = diffs.mean() / diffs.std(ddof=1)

print(f"\nMean rent growth pre-2018:   {mean_pre:.2f}%/yr")
print(f"Mean rent growth post-2018:  {mean_post:.2f}%/yr  (+{mean_post - mean_pre:.2f} pp)")
print()
print(f"Paired t-test:         t={t_stat:.3f},  p={t_p:.4f}  (parametric)")
print(f"Wilcoxon signed-rank:  W={w_stat:.1f},  p={w_p:.4f}  (non-parametric, no normality assumed)")
print(f"Cohen's d = {cohens_d:.3f}  ({'small' if abs(cohens_d)<0.5 else 'medium' if abs(cohens_d)<0.8 else 'large'} effect)")
print()
print("▶  Both parametric and non-parametric tests agree: the acceleration in rent growth")
print(f"   post-2018 is highly significant and represents a {('small' if abs(cohens_d)<0.5 else 'medium' if abs(cohens_d)<0.8 else 'large')}"
      f" effect (d={cohens_d:.2f}).")


Threshold sensitivity (stable vacancy definition):
 Threshold |  Cities stable |  Rents faster |    % |  Binom. p
──────────────────────────────────────────────────────────────
  ≤ 0.3pp    |       8         |       7       | 88%  | 0.0352
  ≤ 0.5pp    |      14         |      13       | 93%  | 0.0009
  ≤ 1.0pp    |      21         |      19       | 90%  | 0.0001

Mean rent growth pre-2018:   2.67%/yr
Mean rent growth post-2018:  4.98%/yr  (+2.31 pp)

Paired t-test:         t=10.096,  p=0.0000  (parametric)
Wilcoxon signed-rank:  W=3.0,  p=0.0000  (non-parametric, no normality assumed)
Cohen's d = 1.843  (large effect)

▶  Both parametric and non-parametric tests agree: the acceleration in rent growth
   post-2018 is highly significant and represents a large effect (d=1.84).


In [15]:
# ── Cell 7: Save summary CSVs ─────────────────────────────────────────────────
summary_dir = CLEAN_ROOT / "summary"
summary_dir.mkdir(exist_ok=True)

# 1. Per-city period comparison (vacancy and rent growth means, pre vs post)
city_period_out = pm_pivot.copy().round(2)
city_period_out.to_csv(summary_dir / "vacancy_rent_period_comparison.csv", index=False)
print("Saved → vacancy_rent_period_comparison.csv")

# 2. Per-city per-period OLS slopes + overall regression table
city_reg.to_csv(summary_dir / "vacancy_rent_regression_by_city_period.csv", index=False)
print("Saved → vacancy_rent_regression_by_city_period.csv")

print("\nDone.")

Saved → vacancy_rent_period_comparison.csv
Saved → vacancy_rent_regression_by_city_period.csv

Done.
